**Problem 11**

A study of length of hospital stay, in days, as a function of age, kind of health insurance, and whether or not the patient died while in the hospital. Length of hospital stay is recorded as a minimum of at least one day. The dataset is taken from:

UCLA Zero-Truncated Dataset

Fit the Zero-Truncated Negative Binomial regression generalized linear model (GLM) to identify the factors associated with hospital length of stay. Interpret the results.

In [2]:
import pandas as pd
import numpy as np

import statsmodels.api as sm

from statsmodels.discrete.truncated_model import TruncatedLFNegativeBinomialP

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error


import urllib.request

In [3]:
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 25.4 MB/s eta 0:00:00


In [4]:
import pyreadstat

In [5]:
url = "https://stats.idre.ucla.edu/stat/data/ztp.dta"

urllib.request.urlretrieve(
    url,
    "ztp.dta"
)

('ztp.dta', <http.client.HTTPMessage at 0x7e15ad235520>)

In [6]:
df, meta = pyreadstat.read_dta(
    "ztp.dta"
)

In [7]:
df.head()

,stay,age,hmo,died
0,4,4,0,0.0
1,9,4,1,0.0
2,3,7,1,1.0
3,9,6,0,0.0
4,1,7,0,1.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1493 entries, 0 to 1492
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   stay    1493 non-null   int64  
 1   age     1493 non-null   int64  
 2   hmo     1493 non-null   int64  
 3   died    1493 non-null   float64
dtypes: float64(1), int64(3)
memory usage: 46.8 KB


In [9]:
#Convert Variables
df['hmo'] = df['hmo'].astype('category')

df['died'] = df['died'].astype('category')

In [10]:
#Check Overdispersion
print(
    "Mean:",
    df['stay'].mean()
)

print(
    "Variance:",
    df['stay'].var()
)

#Variance >> Mean so overdispersion exists

Mean: 9.728734092431345
Variance: 66.14419390578749


In [11]:
#Check Zero Counts
print(
    "Number of zeros:",
    (df['stay'] == 0).sum()
)

Number of zeros: 0


In [12]:
#Prepare Variables

#Response variable:

y = df['stay']

#Predictors:

X = pd.get_dummies(
    df[['age', 'hmo', 'died']],
    drop_first=True
)

#Convert to numeric:

X = X.astype(float)

#Add constant:

X = sm.add_constant(X)

In [13]:
#Fit Zero-Truncated Negative Binomial Model

#Model:stay∼age+hmo+died
ztnb_model = TruncatedLFNegativeBinomialP(
    endog=y,
    exog=X
).fit()

Optimization terminated successfully.
         Current function value: 3.185050
         Iterations: 16
         Function evaluations: 18
         Gradient evaluations: 18


In [14]:
print(ztnb_model.summary())

                    TruncatedLFNegativeBinomialP Regression Results                     
Dep. Variable:                             stay   No. Observations:                 1493
Model:             TruncatedLFNegativeBinomialP   Df Residuals:                     1488
Method:                                     MLE   Df Model:                            3
Date:                          Sun, 10 May 2026   Pseudo R-squ.:                0.003263
Time:                                  13:06:20   Log-Likelihood:                -4755.3
converged:                                 True   LL-Null:                       -4770.8
Covariance Type:                      nonrobust   LLR p-value:                 7.955e-07
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.4083      0.072     33.457      0.000       2.267       2.549
age           -0.0157      0.013     -1.197      0.

# Interpretation of Zero-Truncated Negative Binomial Regression Results

print("""
The Zero-Truncated Negative Binomial regression model was used to analyze hospital length of stay while accounting for overdispersion and the absence of zero-day stays.

The overall model is statistically significant because the Likelihood Ratio p-value is less than 0.05. This indicates that the predictors collectively help explain variation in hospital stay duration.

Interpretation of Predictors:

1. Age
- Coefficient = -0.0157
- P-value = 0.231

Age is not statistically significant because p > 0.05.
There is no strong evidence that age affects hospital stay duration in this model.

2. Insurance Type (hmo_1)
- Coefficient = -0.1471
- P-value = 0.013

Insurance type is statistically significant.
Patients with HMO insurance tend to have shorter hospital stays compared to the reference insurance group.

3. Mortality Status (died_1.0)
- Coefficient = -0.2178
- P-value < 0.001

Mortality status is statistically significant.
Patients in the died = 1 category are associated with shorter expected hospital stays compared to the reference category.

4. Overdispersion Parameter (alpha)
- Alpha = 0.5663
- P-value < 0.001

The significant positive alpha value indicates overdispersion exists in the data.
This supports the use of the Negative Binomial model instead of the Poisson model.

Overall Conclusion:
The analysis suggests that insurance type and mortality status significantly affect hospital length of stay, while age does not show a statistically significant effect. The Zero-Truncated Negative Binomial model is appropriate because the data are overdispersed and exclude zero-day hospital stays.
""")